# Baseline and Lightweight LSTM Improvement

This notebook is designed for a lightweight laptop workflow:

- Build a time-based validation setup first.
- Compare simple statistical baselines before training LSTM.
- Add compact feature-based models.
- Keep LSTM small and optional.
- Generate a competition-style September 2014 submission file.

Raw data is read from `../data/raw/` locally and should not be committed to Git.

In [ ]:
from pathlib import Path
import zipfile
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.errors import PerformanceWarning

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error

warnings.simplefilter('ignore', PerformanceWarning)

PROJECT_ROOT = Path('..').resolve()
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
RAW_ZIP_PATH = RAW_ROOT / 'Purchase Redemption Data.zip'
DEFAULT_EXTRACT_DIR = RAW_ROOT / 'Purchase Redemption Data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)


def find_raw_file(filename):
    matches = sorted(RAW_ROOT.rglob(filename))
    return matches[0] if matches else None


def ensure_raw_data_available():
    balance_path = find_raw_file('user_balance_table.csv')
    if balance_path is not None:
        return balance_path.parent

    if RAW_ZIP_PATH.exists():
        print(f'Extracting raw data zip to: {DEFAULT_EXTRACT_DIR}')
        DEFAULT_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(RAW_ZIP_PATH) as zf:
            zf.extractall(DEFAULT_EXTRACT_DIR)

        balance_path = find_raw_file('user_balance_table.csv')
        if balance_path is not None:
            return balance_path.parent

    raise FileNotFoundError(
        'Could not find user_balance_table.csv. Put the unzipped competition CSV files '
        'under data/raw/, or keep Purchase Redemption Data.zip under data/raw/.'
    )


RAW_DIR = ensure_raw_data_available()
BALANCE_PATH = RAW_DIR / 'user_balance_table.csv'
INTEREST_PATH = RAW_DIR / 'mfd_day_share_interest.csv'
SHIBOR_PATH = RAW_DIR / 'mfd_bank_shibor.csv'

SEED = 2019
USE_EXTERNAL_FEATURES = False
np.random.seed(SEED)

print('RAW_DIR =', RAW_DIR)
print('BALANCE_PATH exists =', BALANCE_PATH.exists())
BALANCE_PATH

## 1. Load Daily Data

The target is daily total purchase and daily total redeem. We aggregate user-level rows to date-level rows.

In [ ]:
def add_date_features(df, date_col='report_date'):
    out = df.copy()
    d = out[date_col]
    out['dayofweek'] = d.dt.dayofweek
    out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
    out['day'] = d.dt.day
    out['month'] = d.dt.month
    out['is_month_start'] = d.dt.is_month_start.astype(int)
    out['is_month_end'] = d.dt.is_month_end.astype(int)
    out['days_to_month_end'] = (d.dt.days_in_month - d.dt.day).astype(int)
    out['is_mid_autumn_2014'] = (d == pd.Timestamp('2014-09-08')).astype(int)
    out['is_mid_autumn_window_2014'] = d.between('2014-09-06', '2014-09-10').astype(int)
    return out


def load_external_features():
    frames = []

    if INTEREST_PATH.exists():
        interest = pd.read_csv(INTEREST_PATH, engine='python')
        interest['report_date'] = pd.to_datetime(interest['mfd_date'].astype(str), format='%Y%m%d')
        interest = interest.drop(columns=['mfd_date'])
        frames.append(interest)

    if SHIBOR_PATH.exists():
        shibor = pd.read_csv(SHIBOR_PATH, engine='python')
        shibor['report_date'] = pd.to_datetime(shibor['mfd_date'].astype(str), format='%Y%m%d')
        shibor = shibor.drop(columns=['mfd_date'])
        frames.append(shibor)

    if not frames:
        return pd.DataFrame({'report_date': []})

    external = frames[0]
    for frame in frames[1:]:
        external = external.merge(frame, on='report_date', how='outer')

    external = external.sort_values('report_date').reset_index(drop=True)
    return external


def load_daily_data():
    usecols = [
        'report_date',
        'total_purchase_amt', 'total_redeem_amt',
        'direct_purchase_amt', 'purchase_bal_amt', 'purchase_bank_amt',
        'consume_amt', 'transfer_amt', 'share_amt'
    ]
    data = pd.read_csv(BALANCE_PATH, usecols=usecols, engine='python')
    data['report_date'] = pd.to_datetime(data['report_date'].astype(str), format='%Y%m%d')

    daily = data.groupby('report_date', as_index=False).sum(numeric_only=True)
    external = load_external_features() if USE_EXTERNAL_FEATURES else pd.DataFrame({'report_date': []})
    if len(external):
        daily = daily.merge(external, on='report_date', how='left')
    daily = daily.sort_values('report_date').reset_index(drop=True)
    daily = add_date_features(daily)
    return daily


daily = load_daily_data()
print(daily.shape)
daily.head()

In [ ]:
daily[['report_date', 'total_purchase_amt', 'total_redeem_amt']].tail()

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(daily['report_date'], daily['total_purchase_amt'], label='purchase')
plt.plot(daily['report_date'], daily['total_redeem_amt'], label='redeem')
plt.title('Daily purchase and redeem')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 2. Validation Metric

The official scoring function is not public, but it is based on daily relative error. We use weighted MAPE-like validation:

- purchase weight: 45%
- redeem weight: 55%

In [ ]:
def relative_abs_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), 1.0)
    return np.abs(y_true - y_pred) / denom


def score_prediction(actual, pred):
    purchase_err = relative_abs_error(actual['purchase'], pred['purchase'])
    redeem_err = relative_abs_error(actual['redeem'], pred['redeem'])
    purchase_proxy_score = 10 * np.maximum(0, 1 - purchase_err / 0.3)
    redeem_proxy_score = 10 * np.maximum(0, 1 - redeem_err / 0.3)
    return {
        'purchase_mape': purchase_err.mean(),
        'redeem_mape': redeem_err.mean(),
        'weighted_mape': 0.45 * purchase_err.mean() + 0.55 * redeem_err.mean(),
        'purchase_max_err': purchase_err.max(),
        'redeem_max_err': redeem_err.max(),
        'proxy_score': 0.45 * purchase_proxy_score.sum() + 0.55 * redeem_proxy_score.sum(),
        'purchase_zero_score_days': int((purchase_err > 0.3).sum()),
        'redeem_zero_score_days': int((redeem_err > 0.3).sum()),
    }


def make_actual(daily, start_date, periods=30):
    start_date = pd.Timestamp(start_date)
    end_date = start_date + pd.Timedelta(days=periods - 1)
    part = daily[daily['report_date'].between(start_date, end_date)].copy()
    return pd.DataFrame({
        'report_date': part['report_date'].values,
        'purchase': part['total_purchase_amt'].values,
        'redeem': part['total_redeem_amt'].values,
    })

## 3. Simple Baselines

These are fast and important. If a neural model cannot beat these, the neural model is not useful yet.

In [ ]:
TARGETS = ['total_purchase_amt', 'total_redeem_amt']


def forecast_last_week(train_daily, start_date, periods=30):
    start_date = pd.Timestamp(start_date)
    rows = []
    history = train_daily[['report_date'] + TARGETS].copy()

    for d in pd.date_range(start_date, periods=periods):
        ref_date = d - pd.Timedelta(days=7)
        ref = history.loc[history['report_date'] == ref_date, TARGETS]
        if len(ref) == 0:
            vals = history[TARGETS].tail(7).mean()
        else:
            vals = ref.iloc[0]
        new_row = {'report_date': d, **vals.to_dict()}
        rows.append(new_row)
        history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)

    out = pd.DataFrame(rows)
    return out.rename(columns={'total_purchase_amt': 'purchase', 'total_redeem_amt': 'redeem'})


def forecast_rolling_mean(train_daily, start_date, window=14, periods=30):
    rows = []
    history = train_daily[['report_date'] + TARGETS].copy()
    for d in pd.date_range(start_date, periods=periods):
        vals = history[TARGETS].tail(window).mean()
        new_row = {'report_date': d, **vals.to_dict()}
        rows.append(new_row)
        history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)
    out = pd.DataFrame(rows)
    return out.rename(columns={'total_purchase_amt': 'purchase', 'total_redeem_amt': 'redeem'})


def forecast_dow_mean(train_daily, start_date, lookback_days=90, periods=30):
    train = train_daily.copy()
    recent = train.tail(lookback_days)
    rows = []
    fallback = recent[TARGETS].mean()
    for d in pd.date_range(start_date, periods=periods):
        dow = d.dayofweek
        group = recent[recent['report_date'].dt.dayofweek == dow]
        vals = group[TARGETS].mean() if len(group) else fallback
        rows.append({'report_date': d, **vals.to_dict()})
    out = pd.DataFrame(rows)
    return out.rename(columns={'total_purchase_amt': 'purchase', 'total_redeem_amt': 'redeem'})


def forecast_dow_median(train_daily, start_date, lookback_days=120, periods=30):
    train = train_daily.copy()
    recent = train.tail(lookback_days)
    rows = []
    fallback = recent[TARGETS].median()
    for d in pd.date_range(start_date, periods=periods):
        dow = d.dayofweek
        group = recent[recent['report_date'].dt.dayofweek == dow]
        vals = group[TARGETS].median() if len(group) else fallback
        rows.append({'report_date': d, **vals.to_dict()})
    out = pd.DataFrame(rows)
    return out.rename(columns={'total_purchase_amt': 'purchase', 'total_redeem_amt': 'redeem'})


def forecast_dow_trend_adjusted(train_daily, start_date, lookback_days=120, periods=30):
    base = forecast_dow_mean(train_daily, start_date, lookback_days=lookback_days, periods=periods)
    recent_mean = train_daily[TARGETS].tail(14).mean()
    older_mean = train_daily[TARGETS].tail(60).head(46).mean()
    ratio = (recent_mean / older_mean.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    ratio = ratio.clip(0.85, 1.15)
    out = base.copy()
    out['purchase'] = out['purchase'] * ratio['total_purchase_amt']
    out['redeem'] = out['redeem'] * ratio['total_redeem_amt']
    return out


def blend_predictions(predictions, weights):
    out = predictions[0][['report_date']].copy()
    total_weight = float(np.sum(weights))
    out['purchase'] = sum(w * p['purchase'].values for p, w in zip(predictions, weights)) / total_weight
    out['redeem'] = sum(w * p['redeem'].values for p, w in zip(predictions, weights)) / total_weight
    return out


def forecast_rule_blend(train_daily, start_date, periods=30):
    preds = [
        forecast_dow_mean(train_daily, start_date, lookback_days=90, periods=periods),
        forecast_dow_median(train_daily, start_date, lookback_days=120, periods=periods),
        forecast_last_week(train_daily, start_date, periods=periods),
    ]
    return blend_predictions(preds, weights=[0.50, 0.30, 0.20])

## 4. Feature-Based Lightweight Model

This recursive Random Forest model is still small enough for a laptop. It uses lag, rolling and date features.

In [ ]:
LAGS = [1, 2, 3, 4, 5, 6, 7, 10, 14, 21, 28, 30, 35]
ROLL_WINDOWS = [3, 7, 14, 21, 30, 60]
EXTERNAL_BASE_FEATURES = [
    'mfd_daily_yield', 'mfd_7daily_yield',
    'Interest_O_N', 'Interest_1_W', 'Interest_2_W', 'Interest_1_M',
    'Interest_3_M', 'Interest_6_M', 'Interest_9_M', 'Interest_1_Y',
]


def add_ts_features(df):
    out = add_date_features(df)
    for target in TARGETS:
        for lag in LAGS:
            out[f'{target}_lag_{lag}'] = out[target].shift(lag)
        for window in ROLL_WINDOWS:
            shifted = out[target].shift(1)
            out[f'{target}_roll_mean_{window}'] = shifted.rolling(window).mean()
            out[f'{target}_roll_std_{window}'] = shifted.rolling(window).std()
    external_cols = [c for c in EXTERNAL_BASE_FEATURES if c in out.columns]
    if external_cols:
        out[external_cols] = out[external_cols].ffill()
        for col in external_cols:
            out[f'{col}_lag_1'] = out[col].shift(1)
            out[f'{col}_lag_7'] = out[col].shift(7)
            out[f'{col}_roll_mean_7'] = out[col].shift(1).rolling(7).mean()
            out[f'{col}_change_1'] = out[col].shift(1) - out[col].shift(2)

    out['net_inflow_lag_1'] = out['total_purchase_amt_lag_1'] - out['total_redeem_amt_lag_1']
    out['purchase_redeem_ratio_lag_1'] = out['total_purchase_amt_lag_1'] / out['total_redeem_amt_lag_1'].replace(0, np.nan)
    out['purchase_week_diff'] = out['total_purchase_amt_lag_1'] - out['total_purchase_amt_lag_7']
    out['redeem_week_diff'] = out['total_redeem_amt_lag_1'] - out['total_redeem_amt_lag_7']
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


DATE_FEATURES = [
    'dayofweek', 'is_weekend', 'day', 'month',
    'is_month_start', 'is_month_end', 'days_to_month_end',
    'is_mid_autumn_2014', 'is_mid_autumn_window_2014',
]


def feature_columns(df):
    # Use only features available at prediction time: calendar features and historical lag/rolling features.
    lag_roll_features = [c for c in df.columns if '_lag_' in c or '_roll_' in c]
    change_features = [c for c in df.columns if c.endswith('_change_1')]
    extra_features = ['net_inflow_lag_1', 'purchase_redeem_ratio_lag_1', 'purchase_week_diff', 'redeem_week_diff']
    extra_features = extra_features + change_features
    return [c for c in DATE_FEATURES + lag_roll_features + extra_features if c in df.columns]


def make_tree_model(model_type, target):
    if model_type == 'random_forest':
        return RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=3,
            random_state=SEED,
            n_jobs=-1,
        )
    if model_type == 'extra_trees':
        return ExtraTreesRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=3,
            random_state=SEED,
            n_jobs=-1,
        )
    if model_type == 'hist_gradient_boosting':
        return HistGradientBoostingRegressor(
            max_iter=120,
            learning_rate=0.04,
            max_leaf_nodes=12,
            l2_regularization=0.05,
            random_state=SEED,
        )
    raise ValueError(model_type)


def train_tree_models(train_daily, model_type='random_forest'):
    train_feat = add_ts_features(train_daily).dropna().reset_index(drop=True)
    features = feature_columns(train_feat)
    models = {}
    for target in TARGETS:
        model = make_tree_model(model_type, target)
        model.fit(train_feat[features], train_feat[target])
        models[target] = model
    return models, features


def forecast_tree(train_daily, start_date, periods=30, model_type='random_forest'):
    models, features = train_tree_models(train_daily, model_type=model_type)
    history_cols = ['report_date'] + TARGETS + [c for c in EXTERNAL_BASE_FEATURES if c in train_daily.columns]
    history = train_daily[history_cols].copy()
    rows = []

    for d in pd.date_range(start_date, periods=periods):
        blank = pd.DataFrame([{'report_date': d, 'total_purchase_amt': np.nan, 'total_redeem_amt': np.nan}])
        tmp = pd.concat([history, blank], ignore_index=True)
        feat_row = add_ts_features(tmp).tail(1)
        x = feat_row[features].fillna(0)
        purchase = models['total_purchase_amt'].predict(x)[0]
        redeem = models['total_redeem_amt'].predict(x)[0]
        purchase = max(0, purchase)
        redeem = max(0, redeem)
        new_row = {'report_date': d, 'total_purchase_amt': purchase, 'total_redeem_amt': redeem}
        rows.append(new_row)
        history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)

    out = pd.DataFrame(rows)
    return out.rename(columns={'total_purchase_amt': 'purchase', 'total_redeem_amt': 'redeem'})


def forecast_rf(train_daily, start_date, periods=30):
    return forecast_tree(train_daily, start_date, periods=periods, model_type='random_forest')


def forecast_extra_trees(train_daily, start_date, periods=30):
    return forecast_tree(train_daily, start_date, periods=periods, model_type='extra_trees')


def forecast_hgb(train_daily, start_date, periods=30):
    return forecast_tree(train_daily, start_date, periods=periods, model_type='hist_gradient_boosting')


def forecast_model_blend(train_daily, start_date, periods=30):
    preds = [
        forecast_rf(train_daily, start_date, periods=periods),
        forecast_extra_trees(train_daily, start_date, periods=periods),
        forecast_dow_mean(train_daily, start_date, lookback_days=90, periods=periods),
    ]
    return blend_predictions(preds, weights=[0.45, 0.25, 0.30])

## 5. Time-Based Backtest

Each split simulates using historical data to forecast the next 30 days.

In [ ]:
VALIDATION_STARTS = [
    '2014-05-01',
    '2014-06-01',
    '2014-07-01',
    '2014-08-01',
]


def run_backtest(daily, validation_starts=VALIDATION_STARTS):
    records = []
    methods = {
        'last_week': lambda train, start: forecast_last_week(train, start),
        'rolling_mean_14': lambda train, start: forecast_rolling_mean(train, start, window=14),
        'dow_mean_90': lambda train, start: forecast_dow_mean(train, start, lookback_days=90),
        'dow_median_120': lambda train, start: forecast_dow_median(train, start, lookback_days=120),
        'dow_trend_adjusted': lambda train, start: forecast_dow_trend_adjusted(train, start, lookback_days=120),
        'rule_blend': lambda train, start: forecast_rule_blend(train, start),
        'random_forest': lambda train, start: forecast_rf(train, start),
        'extra_trees': lambda train, start: forecast_extra_trees(train, start),
        'hist_gradient_boosting': lambda train, start: forecast_hgb(train, start),
        'model_blend': lambda train, start: forecast_model_blend(train, start),
    }

    for start in validation_starts:
        start_ts = pd.Timestamp(start)
        train = daily[daily['report_date'] < start_ts].copy()
        actual = make_actual(daily, start_ts)
        for name, fn in methods.items():
            pred = fn(train, start_ts)
            metrics = score_prediction(actual, pred)
            records.append({'start_date': start, 'method': name, **metrics})
    return pd.DataFrame(records).sort_values(['weighted_mape', 'start_date'])


BLEND_SEARCH_METHODS = ['extra_trees', 'dow_median_120', 'dow_mean_90']
BLEND_FORECASTERS = {
    'extra_trees': lambda train, start: forecast_extra_trees(train, start),
    'dow_median_120': lambda train, start: forecast_dow_median(train, start, lookback_days=120),
    'dow_mean_90': lambda train, start: forecast_dow_mean(train, start, lookback_days=90),
}


def search_blend_weights(daily, validation_starts=VALIDATION_STARTS, step=0.1):
    prediction_cache = {}
    actual_cache = {}

    for start in validation_starts:
        start_ts = pd.Timestamp(start)
        train = daily[daily['report_date'] < start_ts].copy()
        actual_cache[start] = make_actual(daily, start_ts)
        for method in BLEND_SEARCH_METHODS:
            prediction_cache[(start, method)] = BLEND_FORECASTERS[method](train, start_ts)

    records = []
    grid = np.arange(0, 1 + step / 2, step)
    for w0 in grid:
        for w1 in grid:
            w2 = 1 - w0 - w1
            if abs(w2) < 1e-9:
                w2 = 0.0
            if w2 < 0:
                continue
            weights = np.array([w0, w1, w2], dtype=float)
            if weights.sum() <= 0:
                continue

            scores = []
            for start in validation_starts:
                preds = [prediction_cache[(start, method)] for method in BLEND_SEARCH_METHODS]
                pred = blend_predictions(preds, weights=weights)
                scores.append(score_prediction(actual_cache[start], pred)['weighted_mape'])

            records.append({
                'extra_trees_weight': weights[0],
                'dow_median_120_weight': weights[1],
                'dow_mean_90_weight': weights[2],
                'mean_weighted_mape': float(np.mean(scores)),
            })

    return pd.DataFrame(records).sort_values('mean_weighted_mape').reset_index(drop=True)


def forecast_weight_search_blend(train_daily, start_date, periods=30, weights=None):
    if weights is None:
        weights = BEST_BLEND_WEIGHTS
    preds = [BLEND_FORECASTERS[method](train_daily, start_date) for method in BLEND_SEARCH_METHODS]
    weight_values = [weights[method] for method in BLEND_SEARCH_METHODS]
    return blend_predictions(preds, weight_values)


def run_one_method_backtest(daily, method_name, forecast_fn, validation_starts=VALIDATION_STARTS):
    records = []
    for start in validation_starts:
        start_ts = pd.Timestamp(start)
        train = daily[daily['report_date'] < start_ts].copy()
        actual = make_actual(daily, start_ts)
        pred = forecast_fn(train, start_ts)
        metrics = score_prediction(actual, pred)
        records.append({'start_date': start, 'method': method_name, **metrics})
    return pd.DataFrame(records)


def apply_calendar_adjustment(
    pred,
    weekend_redeem_factor=1.0,
    month_end_purchase_factor=1.0,
    month_start_purchase_factor=1.0,
    month_start_redeem_factor=1.0,
):
    out = pred.copy()
    dates = out['report_date']
    weekend_mask = dates.dt.dayofweek >= 5
    month_end_mask = dates.dt.day >= 25
    month_start_mask = dates.dt.day <= 5
    out.loc[weekend_mask, 'redeem'] = out.loc[weekend_mask, 'redeem'] * weekend_redeem_factor
    out.loc[month_end_mask, 'purchase'] = out.loc[month_end_mask, 'purchase'] * month_end_purchase_factor
    out.loc[month_start_mask, 'purchase'] = out.loc[month_start_mask, 'purchase'] * month_start_purchase_factor
    out.loc[month_start_mask, 'redeem'] = out.loc[month_start_mask, 'redeem'] * month_start_redeem_factor
    return out


def search_calendar_adjustments(daily, validation_starts=VALIDATION_STARTS):
    weekend_redeem_grid = [0.90, 0.95, 1.00]
    month_end_purchase_grid = [0.90, 0.95, 1.00]
    month_start_purchase_grid = [0.80, 0.90, 1.00]
    month_start_redeem_grid = [0.80, 0.90, 1.00]
    base_cache = {}
    actual_cache = {}

    for start in validation_starts:
        start_ts = pd.Timestamp(start)
        train = daily[daily['report_date'] < start_ts].copy()
        actual_cache[start] = make_actual(daily, start_ts)
        base_cache[start] = forecast_weight_search_blend(train, start_ts)

    records = []
    for weekend_redeem_factor in weekend_redeem_grid:
        for month_end_purchase_factor in month_end_purchase_grid:
            for month_start_purchase_factor in month_start_purchase_grid:
                for month_start_redeem_factor in month_start_redeem_grid:
                    scores = []
                    for start in validation_starts:
                        pred = apply_calendar_adjustment(
                            base_cache[start],
                            weekend_redeem_factor=weekend_redeem_factor,
                            month_end_purchase_factor=month_end_purchase_factor,
                            month_start_purchase_factor=month_start_purchase_factor,
                            month_start_redeem_factor=month_start_redeem_factor,
                        )
                        scores.append(score_prediction(actual_cache[start], pred)['weighted_mape'])

                    records.append({
                        'weekend_redeem_factor': weekend_redeem_factor,
                        'month_end_purchase_factor': month_end_purchase_factor,
                        'month_start_purchase_factor': month_start_purchase_factor,
                        'month_start_redeem_factor': month_start_redeem_factor,
                        'mean_weighted_mape': float(np.mean(scores)),
                    })

    return pd.DataFrame(records).sort_values('mean_weighted_mape').reset_index(drop=True)


def forecast_calendar_adjusted_blend(train_daily, start_date, periods=30, factors=None):
    if factors is None:
        factors = BEST_CALENDAR_FACTORS
    base_pred = forecast_weight_search_blend(train_daily, start_date, periods=periods)
    return apply_calendar_adjustment(
        base_pred,
        weekend_redeem_factor=factors['weekend_redeem_factor'],
        month_end_purchase_factor=factors['month_end_purchase_factor'],
        month_start_purchase_factor=factors['month_start_purchase_factor'],
        month_start_redeem_factor=factors['month_start_redeem_factor'],
    )


blend_search_result = search_blend_weights(daily, step=0.1)
best_blend_row = blend_search_result.iloc[0]
BEST_BLEND_WEIGHTS = {
    'extra_trees': best_blend_row['extra_trees_weight'],
    'dow_median_120': best_blend_row['dow_median_120_weight'],
    'dow_mean_90': best_blend_row['dow_mean_90_weight'],
}

backtest_result = run_backtest(daily)
weighted_blend_backtest = run_one_method_backtest(daily, 'weight_search_blend', forecast_weight_search_blend)
calendar_adjustment_result = search_calendar_adjustments(daily)
best_calendar_row = calendar_adjustment_result.iloc[0]
BEST_CALENDAR_FACTORS = {
    'weekend_redeem_factor': best_calendar_row['weekend_redeem_factor'],
    'month_end_purchase_factor': best_calendar_row['month_end_purchase_factor'],
    'month_start_purchase_factor': best_calendar_row['month_start_purchase_factor'],
    'month_start_redeem_factor': best_calendar_row['month_start_redeem_factor'],
}
calendar_adjusted_backtest = run_one_method_backtest(daily, 'calendar_adjusted_blend', forecast_calendar_adjusted_blend)
backtest_result = pd.concat([backtest_result, weighted_blend_backtest], ignore_index=True)
backtest_result = pd.concat([backtest_result, calendar_adjusted_backtest], ignore_index=True)
backtest_result = backtest_result.sort_values(['weighted_mape', 'start_date']).reset_index(drop=True)

print('Best blend weights:', BEST_BLEND_WEIGHTS)
print('Blend search top 5:')
print(blend_search_result.head())
print('Best calendar factors:', BEST_CALENDAR_FACTORS)
print('Calendar adjustment search top 5:')
print(calendar_adjustment_result.head())
backtest_result

## 6. Optional Lightweight LSTM

The previous notebook used shape `(samples, 1, look_back)`. A more natural sequence shape is `(samples, look_back, 1)`.

Keep `RUN_LSTM = False` until the baselines work. Set it to `True` when you want to test the compact LSTM.

In [ ]:
RUN_LSTM = False


def make_lstm_dataset(values, look_back=30, horizon=30):
    x, y = [], []
    for i in range(len(values) - look_back - horizon + 1):
        x.append(values[i:i + look_back])
        y.append(values[i + look_back:i + look_back + horizon])
    x = np.asarray(x, dtype='float32').reshape(-1, look_back, 1)
    y = np.asarray(y, dtype='float32')
    return x, y


def forecast_lstm_one_target(train_values, look_back=30, horizon=30, epochs=50):
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import MinMaxScaler

    tf.keras.backend.clear_session()
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(np.asarray(train_values).reshape(-1, 1)).reshape(-1)
    x, y = make_lstm_dataset(scaled, look_back=look_back, horizon=horizon)

    model = Sequential([
        LSTM(24, input_shape=(look_back, 1)),
        Dense(16, activation='relu'),
        Dense(horizon),
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(
        x, y,
        epochs=epochs,
        batch_size=8,
        verbose=0,
        callbacks=[EarlyStopping(monitor='loss', patience=8, restore_best_weights=True)],
    )
    test_x = scaled[-look_back:].reshape(1, look_back, 1)
    pred_scaled = model.predict(test_x, verbose=0).reshape(-1, 1)
    pred = scaler.inverse_transform(pred_scaled).reshape(-1)

    tf.keras.backend.clear_session()
    gc.collect()
    return np.maximum(pred, 0)


def forecast_lstm(train_daily, start_date, look_back=30, horizon=30, epochs=50):
    purchase = forecast_lstm_one_target(train_daily['total_purchase_amt'].values, look_back, horizon, epochs)
    redeem = forecast_lstm_one_target(train_daily['total_redeem_amt'].values, look_back, horizon, epochs)
    return pd.DataFrame({
        'report_date': pd.date_range(start_date, periods=horizon),
        'purchase': purchase,
        'redeem': redeem,
    })


if RUN_LSTM:
    start = pd.Timestamp('2014-08-01')
    train = daily[daily['report_date'] < start].copy()
    actual = make_actual(daily, start)
    pred_lstm = forecast_lstm(train, start, look_back=30, horizon=30, epochs=50)
    print(score_prediction(actual, pred_lstm))
    pred_lstm.head()

## 7. Error Diagnostics\n\nInspect daily validation errors for the current best method before deciding on more rules.

In [ ]:
DIAGNOSTIC_METHODS = {
    'calendar_adjusted_blend': forecast_calendar_adjusted_blend,
    'weight_search_blend': forecast_weight_search_blend,
    'extra_trees': forecast_extra_trees,
    'dow_median_120': lambda train, start: forecast_dow_median(train, start, lookback_days=120),
}


def make_error_detail(actual, pred, method, start_date):
    detail = actual.merge(pred, on='report_date', suffixes=('_actual', '_pred'))
    detail['method'] = method
    detail['start_date'] = start_date
    detail['dayofweek'] = detail['report_date'].dt.dayofweek
    detail['is_weekend'] = (detail['dayofweek'] >= 5).astype(int)
    detail['day'] = detail['report_date'].dt.day
    detail['is_month_start'] = detail['report_date'].dt.is_month_start.astype(int)
    detail['is_month_end'] = detail['report_date'].dt.is_month_end.astype(int)
    detail['purchase_error'] = detail['purchase_pred'] - detail['purchase_actual']
    detail['redeem_error'] = detail['redeem_pred'] - detail['redeem_actual']
    detail['purchase_abs_pct_error'] = relative_abs_error(detail['purchase_actual'], detail['purchase_pred'])
    detail['redeem_abs_pct_error'] = relative_abs_error(detail['redeem_actual'], detail['redeem_pred'])
    detail['weighted_abs_pct_error'] = 0.45 * detail['purchase_abs_pct_error'] + 0.55 * detail['redeem_abs_pct_error']
    return detail


def run_error_diagnostics(daily, methods=DIAGNOSTIC_METHODS, validation_starts=VALIDATION_STARTS):
    details = []
    for start in validation_starts:
        start_ts = pd.Timestamp(start)
        train = daily[daily['report_date'] < start_ts].copy()
        actual = make_actual(daily, start_ts)
        for method, fn in methods.items():
            pred = fn(train, start_ts)
            details.append(make_error_detail(actual, pred, method, start))
    return pd.concat(details, ignore_index=True)


error_detail = run_error_diagnostics(daily)

worst_days = (
    error_detail
    .sort_values('weighted_abs_pct_error', ascending=False)
    .loc[:, [
        'start_date', 'method', 'report_date', 'dayofweek', 'is_weekend',
        'purchase_actual', 'purchase_pred', 'purchase_abs_pct_error',
        'redeem_actual', 'redeem_pred', 'redeem_abs_pct_error',
        'weighted_abs_pct_error'
    ]]
    .head(15)
)

weekday_summary = (
    error_detail
    .groupby(['method', 'dayofweek'], as_index=False)[
        ['purchase_abs_pct_error', 'redeem_abs_pct_error', 'weighted_abs_pct_error']
    ]
    .mean()
    .sort_values(['method', 'weighted_abs_pct_error'])
)

weekend_summary = (
    error_detail
    .groupby(['method', 'is_weekend'], as_index=False)[
        ['purchase_abs_pct_error', 'redeem_abs_pct_error', 'weighted_abs_pct_error']
    ]
    .mean()
    .sort_values(['method', 'is_weekend'])
)

month_position_summary = (
    error_detail
    .assign(
        month_position=np.select(
            [error_detail['day'] <= 5, error_detail['day'] >= 25],
            ['month_start', 'month_end'],
            default='middle'
        )
    )
    .groupby(['method', 'month_position'], as_index=False)[
        ['purchase_abs_pct_error', 'redeem_abs_pct_error', 'weighted_abs_pct_error']
    ]
    .mean()
    .sort_values(['method', 'weighted_abs_pct_error'])
)

print('Worst validation days:')
print(worst_days.to_string(index=False))
print('\nWeekend summary:')
print(weekend_summary.to_string(index=False))
print('\nMonth position summary:')
print(month_position_summary.to_string(index=False))



## 7. Generate September 2014 Submission

Choose one method after looking at validation results. Start with a strong baseline or Random Forest, then compare optional LSTM.

In [ ]:
METHOD_FOR_SUBMISSION = 'calendar_adjusted_blend'
# options: last_week, rolling_mean_14, dow_mean_90, dow_median_120,
# dow_trend_adjusted, rule_blend, random_forest, extra_trees,
# hist_gradient_boosting, model_blend, weight_search_blend,
# calendar_adjusted_blend, lstm
SUBMISSION_START = pd.Timestamp('2014-09-01')

if METHOD_FOR_SUBMISSION == 'last_week':
    submission = forecast_last_week(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'rolling_mean_14':
    submission = forecast_rolling_mean(daily, SUBMISSION_START, window=14)
elif METHOD_FOR_SUBMISSION == 'dow_mean_90':
    submission = forecast_dow_mean(daily, SUBMISSION_START, lookback_days=90)
elif METHOD_FOR_SUBMISSION == 'dow_median_120':
    submission = forecast_dow_median(daily, SUBMISSION_START, lookback_days=120)
elif METHOD_FOR_SUBMISSION == 'dow_trend_adjusted':
    submission = forecast_dow_trend_adjusted(daily, SUBMISSION_START, lookback_days=120)
elif METHOD_FOR_SUBMISSION == 'rule_blend':
    submission = forecast_rule_blend(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'random_forest':
    submission = forecast_rf(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'extra_trees':
    submission = forecast_extra_trees(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'hist_gradient_boosting':
    submission = forecast_hgb(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'model_blend':
    submission = forecast_model_blend(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'weight_search_blend':
    submission = forecast_weight_search_blend(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'calendar_adjusted_blend':
    submission = forecast_calendar_adjusted_blend(daily, SUBMISSION_START)
elif METHOD_FOR_SUBMISSION == 'lstm':
    submission = forecast_lstm(daily, SUBMISSION_START, look_back=30, horizon=30, epochs=50)
else:
    raise ValueError(METHOD_FOR_SUBMISSION)

submission = submission.copy()
submission['report_date'] = submission['report_date'].dt.strftime('%Y%m%d').astype(int)
submission['purchase'] = np.maximum(submission['purchase'], 0).round().astype(int)
submission['redeem'] = np.maximum(submission['redeem'], 0).round().astype(int)
submission = submission[['report_date', 'purchase', 'redeem']]

submission_path = OUTPUT_DIR / f'submit_{METHOD_FOR_SUBMISSION}.csv'
submission.to_csv(submission_path, index=False)
print(submission_path)
submission